In [6]:
import torch
import torch.nn.functional as F
import joblib
import pandas as pd
from PIL import Image
from torchvision import transforms
import argparse
import logging
import os
import matplotlib.pyplot as plt
import warnings

# Suppress messy warnings for a clean terminal output
warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format='%(message)s')
logger = logging.getLogger(__name__)

def generate_visual_receipt(image_path, make, model, year, severity, confidence, estimated_cost):
    """Generates a professional visual receipt combining the image and the AI prediction."""
    try:
        img = Image.open(image_path)
        fig, ax = plt.subplots(figsize=(10, 6))
        
        # Display the car image
        ax.imshow(img)
        ax.axis('off')
        
        # Build the receipt text
        receipt_text = (
            f" AUTOMATED CLAIM ASSESSMENT \n"
            f" {'-'*30}\n"
            f" Vehicle:     {year} {make} {model}\n"
            f" AI Vision:   {severity} Damage\n"
            f" Confidence:  {confidence:.1f}%\n"
            f" {'-'*30}\n"
            f" Est. Quote:  ${estimated_cost:,.2f} \n"
        )
        
        # Add a styled text box to the image
        props = dict(boxstyle='round,pad=1', facecolor='#222831', alpha=0.85, edgecolor='#00adb5', lw=2)
        ax.text(0.05, 0.95, receipt_text, transform=ax.transAxes, fontsize=14,
                verticalalignment='top', color='white', bbox=props, family='monospace')
        
        # Save locally
        os.makedirs('reports', exist_ok=True)
        filename = f"reports/Quote_{make}_{model}_{year}.png"
        plt.savefig(filename, bbox_inches='tight', dpi=150)
        logger.info(f"\n[SUCCESS] Visual Receipt generated and saved to: ./{filename}")
        plt.close()
        
    except Exception as e:
        logger.error(f"[ERROR] Failed to generate visual report: {e}")

def run_pipeline(image_path, make, model, year):
    logger.info(f"\n=> Initiating AI Pipeline for {year} {make} {model}...")
    
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    
    try:
        cv_model = torch.load('saved_models/damage_classifier_cnn.pth', map_location=device)
        cv_model.eval() 
        ml_pipeline = joblib.load('saved_models/pricing_xgboost_pipeline.pkl')
    except Exception as e:
        logger.error(f"[ERROR] Models not found. Did you run the notebooks? Details: {e}")
        return

    # 1. Vision Processing
    logger.info("=> Analyzing damage via ResNet-50...")
    preprocess = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    
    img = Image.open(image_path).convert('RGB')
    input_tensor = preprocess(img).unsqueeze(0).to(device)
    
    class_names = ['01-minor', '02-moderate', '03-severe']
    with torch.no_grad():
        output = cv_model(input_tensor)
        probs = F.softmax(output, dim=1)[0] * 100
        
    _, predicted_idx = torch.max(output, 1)
    severity_label = class_names[predicted_idx.item()]
    confidence = probs[predicted_idx.item()].item()
    clean_severity = severity_label.split('-')[1].title()
    
    # 2. Tabular Processing
    logger.info("=> Calculating localized financial parameters via XGBoost...")
    input_data = pd.DataFrame([{
        'auto_make': make, 'auto_model': model, 
        'auto_year': year, 'incident_severity': severity_label
    }])
    
    estimated_cost = ml_pipeline.predict(input_data)[0]
    
    # 3. Output Generation
    generate_visual_receipt(image_path, make, model, year, clean_severity, confidence, estimated_cost)

if __name__ == "__main__":import torch
import torch.nn.functional as F
import joblib
import pandas as pd
from PIL import Image
from torchvision import transforms
import logging
import os
import matplotlib.pyplot as plt
import warnings

# Suppress messy warnings for a clean terminal output
warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format='%(message)s')
logger = logging.getLogger(__name__)

def generate_visual_receipt(image_path, make, model, year, severity, confidence, estimated_cost):
    """Generates a professional visual receipt combining the image and the AI prediction."""
    try:
        img = Image.open(image_path)
        fig, ax = plt.subplots(figsize=(10, 6))
        
        # Display the car image
        ax.imshow(img)
        ax.axis('off')
        
        # Build the receipt text
        receipt_text = (
            f" AUTOMATED CLAIM ASSESSMENT \n"
            f" {'-'*30}\n"
            f" Vehicle:     {year} {make} {model}\n"
            f" AI Vision:   {severity} Damage\n"
            f" Confidence:  {confidence:.1f}%\n"
            f" {'-'*30}\n"
            f" Est. Quote:  ${estimated_cost:,.2f} \n"
        )
        
        # Add a styled text box to the image
        props = dict(boxstyle='round,pad=1', facecolor='#222831', alpha=0.85, edgecolor='#00adb5', lw=2)
        ax.text(0.05, 0.95, receipt_text, transform=ax.transAxes, fontsize=14,
                verticalalignment='top', color='white', bbox=props, family='monospace')
        
        # Save locally
        os.makedirs('reports', exist_ok=True)
        filename = f"reports/Quote_{make}_{model}_{year}.png"
        plt.savefig(filename, bbox_inches='tight', dpi=150)
        logger.info(f"\n[SUCCESS] Visual Receipt generated and saved to: ./{filename}")
        plt.close()
        
    except Exception as e:
        logger.error(f"[ERROR] Failed to generate visual report: {e}")

def run_pipeline(image_path, make, model, year):
    logger.info(f"\n=> Initiating AI Pipeline for {year} {make} {model}...")
    
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    
    try:
        # THE FIX 1: Added weights_only=False for PyTorch 2.6 security
        cv_model = torch.load('saved_models/damage_classifier_cnn.pth', map_location=device, weights_only=False)
        cv_model.eval() 
        
        # THE FIX 2: Pointing to the new Scikit-Learn pipeline instead of XGBoost
        ml_pipeline = joblib.load('saved_models/pricing_sklearn_pipeline.pkl')
        
    except Exception as e:
        logger.error(f"[ERROR] Models not found. Details: {e}")
        return

    # 1. Vision Processing
    logger.info("=> Analyzing damage via ResNet-50...")
    preprocess = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    
    try:
        img = Image.open(image_path).convert('RGB')
    except Exception as e:
        logger.error(f"[ERROR] Could not open image at {image_path}. Details: {e}")
        return

    input_tensor = preprocess(img).unsqueeze(0).to(device)
    
    class_names = ['01-minor', '02-moderate', '03-severe']
    with torch.no_grad():
        output = cv_model(input_tensor)
        probs = F.softmax(output, dim=1)[0] * 100
        
    _, predicted_idx = torch.max(output, 1)
    severity_label = class_names[predicted_idx.item()]
    confidence = probs[predicted_idx.item()].item()
    clean_severity = severity_label.split('-')[1].title()
    
    # 2. Tabular Processing
    logger.info("=> Calculating localized financial parameters...")
    
    # Needs to match the input format expected by Scikit-Learn pipeline
    input_data = pd.DataFrame([{
        'auto_make': make, 
        'auto_model': model, 
        'auto_year': year, 
        'incident_severity': severity_label
    }])
    
    estimated_cost = ml_pipeline.predict(input_data)[0]
    
    # 3. Output Generation
    generate_visual_receipt(image_path, make, model, year, clean_severity, confidence, estimated_cost)


if __name__ == "__main__":
    
    TEST_IMAGE_PATH = "data3a 2/validation/02-moderate/0010.jpeg" 
    
    TEST_MAKE = "Kia"
    TEST_MODEL = "Sonet"
    TEST_YEAR = 2024
    
    run_pipeline(TEST_IMAGE_PATH, TEST_MAKE, TEST_MODEL, TEST_YEAR)
    run_pipeline("data3a 2/validation/03-severe/0005.jpeg", "KIA", "SONNET", 2024)


=> Initiating AI Pipeline for 2024 Kia Sonet...
=> Analyzing damage via ResNet-50...
=> Calculating localized financial parameters...

[SUCCESS] Visual Receipt generated and saved to: ./reports/Quote_Kia_Sonet_2024.png

=> Initiating AI Pipeline for 2024 KIA SONNET...
=> Analyzing damage via ResNet-50...
=> Calculating localized financial parameters...

[SUCCESS] Visual Receipt generated and saved to: ./reports/Quote_KIA_SONNET_2024.png
